In [4]:
from dotenv import load_dotenv
import os
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.agents import Tool, initialize_agent, AgentType
from langchain.memory import ConversationBufferMemory

load_dotenv()
google_api_key=os.getenv('GOOGLE_API_KEY')

llm=ChatGoogleGenerativeAI(model='gemini-2.5-flash',temperature=0,convert_system_message_to_human=True)

with open('sample.txt','r',encoding='utf-8') as f:
    text_data=f.read()

splitter=CharacterTextSplitter(separator='\n',chunk_size=300,chunk_overlap=50)
texts=splitter.split_text(text_data)
embeddings=GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-2-preview')
vectorstore=FAISS.from_texts(texts,embeddings)
retriever=vectorstore.as_retriever()

qa_chain=RetrievalQA.from_chain_type(llm=llm,retriever=retriever)

qa_tool=Tool(
    name='Simple QA',
    func=qa_chain.invoke,
    description='A basic LLM that answers users queries clearly')

memory=ConversationBufferMemory(memory_key='chat_history',return_messages=True)

agent=initialize_agent(
    tools=[qa_tool],
    llm=llm,
    memory=memory,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)

print("First Question")
res1 = agent.invoke("What is LangChain?")
print("Answer:" , res1)

print("\nFollow-up")
res2 = agent.invoke("Who created it?")
print("Answer:" , res2)

print("\nCombined Reasoning")
res3 = agent.invoke("Explain LangChain's use in AI workflows.")
print("Answer:" , res3)

C:\Users\DELL\AppData\Local\Temp\ipykernel_4348\982114719.py:31: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory=ConversationBufferMemory(memory_key='chat_history',return_messages=True)
C:\Users\DELL\AppData\Local\Temp\ipykernel_4348\982114719.py:33: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent=initialize_agen

First Question


> Entering new AgentExecutor chain...
Thought: Do I need to use a tool? Yes
Action: Simple QA
Action Input: What is LangChain?
Observation: {'query': 'What is LangChain?', 'result': "LangChain is a framework for building applications with LLMs. It supports RAG, agents, memory, tools, and more. It's commonly used in chatbots, document Q&A, and AI workflow. LangChain was created by Harrison Chase."}
Thought:Do I need to use a tool? No
AI: LangChain is a framework designed for building applications that use large language models (LLMs). It offers features like Retrieval Augmented Generation (RAG), agents, memory, and tools, making it useful for creating chatbots, document question-answering systems, and various AI workflows. It was developed by Harrison Chase.

> Finished chain.
Answer: {'input': 'What is LangChain?', 'chat_history': [HumanMessage(content='What is LangChain?', additional_kwargs={}, response_metadata={}), AIMessage(content='LangChain is a framework designe